# Model #

I try to put down a model. I will use the xor-perceptron model as a base and modify that.

first create the lattice, N by N, with Nsol = N**2

I should create a class for the cell object and then the network class is formed by cell objects and when the network is created, each network is associated with a position in the lattice.

Note: for now I'm only considering one possible link between each pair neuron for each direction

In [5]:
import jax
from jax import random as jrd
from jax import numpy as jnp
from jax import debug as jdb
from jax import lax
import graph_tool as gt
from graph_tool.all import *
import copy
from typing import NamedTuple

In [2]:
# parameters of the simulation
par = {'key': jrd.key(1634),    # key for random generation
       'N': 10,
       'int_range': 1,             # interaction range
       'p_self_link': 0.5,
       'length_powerlaw': 1,         # exponent of the wire length power law (DA DEFINIRE!!!)
       'target_set': [0.0,1.0,1.0,0.0],
       'input_set':[[0,0],[0,1],[1,0],[1,1]],
       'N_sol': 10}

In [ ]:
# define a quick function for generating a new random key from par['key'] and update par['key']
def gen_key(par:dict) -> tuple[jax.KeyArray,dict]:  # JAXXED!
    """Function to generate a new key, used for random generation in JAX, and update the current key
       present in par.

    Args:
        par (dict): dictionary of parameters, including also the current key.

    Returns:
        tuple[jax.KeyArray,dict]: [new key, updated par dictionary]
    """
    key, subkey = jrd.split(par['key'])     # generate a random key by splitting the key in par
    par['key'] = key                        # update the key in par
    return subkey, par                      # I have to return also par, otherwise, bc of JIT's rules, par['key'] won't update

# PROVO A DEFINIRE UNA NAMED TUPLE CONTENENTE TUTTI I PARAMETRI DEL NETWORK (?)
class NetworkParams(NamedTuple):
    """NamedTuple class for the parameters of the network."""
    J: jnp.ndarray          # weight matrix
    C: jnp.ndarray          # connectivity matrix
    B: jnp.ndarray          # bias matrix
    G: Graph                # Graph of the network (Graph_tool)
    state: jnp.ndarray      # values (0 or 1) of the cells in the network
    activation: jnp.ndarray # array of the activation functions of the single cells in the network
    N: int = 0              # side lenght of the squared lattice (number of cells = N**2)
    fitness: int = None     # fitness value of the network
    
# ORA DEFINISCO LA CLASSE NETWORK, CHE è DI FATTO SOLO UN NAMESPACE PER I VARI METODI
# CHE USO SUL SINGOLO NETWORK, I.E. 
# - __INIT__(PAR,NETPAR): CREA UN NETWORK RANDOM DATI I PARAMETRI PAR E RITORNA I PARAMETRI
#   DEL NETWORK (NETPAR) AGGIORNATI
# - COMPUTE_FITNESS(PAR,NETPAR): CALCOLA LA FITNESS DEL NETWORK ATTRAVERSO IL METODO FF() E
#   RITORNA TALE FITNESS.
# - FF(INPUT(S),NETPAR): FA IL 'FEED-FORWARD' DEL NETWORK DATO UN SET DI INPUT E RITORNA 
#   LO STATE (OSSIA L'ARRAY DEGLI STATI 0 O 1 DELLE SINGOLE CELL)

class Network():
    
    def __init__(self,NetPar:NetworkParams,par:dict) -> tuple[NetworkParams,dict]:
        """Generate a random network given the parameters par by updating the 
           NetPar object.

        Args:
            NetPar (NetworkParams): NamedTuple of the parameters of the network.
            par (dict): simulation/generation parameters.

        Returns:
            tuple[NetworkParams,dict]: returns the updated network's parameters and also par 
            since we updated par['key'].
        """
        NetPar.N = par['N']
        subkey, par = gen_key(par)                                      # generate the subkey for the random generation in the following line
        NetPar.J = jrd.uniform(subkey,shape=(NetPar.N**2,NetPar.N**2))                 # uniformly populate the weights in the weight matrix J  
        subkey, par = gen_key(par)                                      # generate the subkey for the random generation in the following line
        NetPar.B = jrd.normal(subkey,shape=(NetPar.N**2,NetPar.N**2))                  # extract from a normal distribution centered in 0 with st. dv. = 1 the biases (spero vada bene fatto così)
        subkey, par = gen_key(par)
        prob_matrix = jnp.where(jnp.eye(NetPar.N**2, dtype=bool),        # define a matrix for the link probability for C
                    par['p_NetPar_link'], 1 / NetPar.D)
        NetPar.C = jnp.where(jrd.bernoulli(subkey, prob_matrix),1,0)    # Generate C  
        NetPar.G = Graph(jnp.column_stack(jnp.nonzero(self.C)))         # Generate the Graph given the connectivity matrix
        NetPar.fitness = self.compute_fitness(par)                      # Compute the fitness value of the network
        return NetPar, par
    
    def compute_fitness(self,NetPar:NetworkParams,par:dict,verb:int=0) -> float:
        # DEFINISCI QUESTA FUNZIONE!!
        if NetPar.fitness is None:        # I need this check, bc otherwise I risk adding fitness over fitness
            for i,input in enumerate(par['input_set']):
            lax.fori_loop(0,len(par['input_set']),self.compute_cost())
            fitness /= len(par['input_set'])    
        return fitness
    
    def compute_cost(self,i,input,verb,NetPar,par)->float:
        fitness = 0.
        output = NetPar.ff(input)
        if verb > 0:
            jdb.print('Input: {input} -> {output}',input=input,output=output)
        norm_factor = NetPar.G.num_vertices()**2-NetPar.G.num_vertices()                # normalize by N(N-1)
        target_dist = (par['target_set'][i] - output)**2                            # square distance between network output and target (theoretical) output
        volume_cost = (jnp.sum(NetPar.C.reshape(-1))/norm_factor)**2                     # average wiring volume cost (i.e. # of links)
        length_cost = (jnp.sum((NetPar.C.reshape(-1) * NetPar.D.reshape(-1))**par['length_powerlaw'])
                        /norm_factor)**2                                      # average wiring length cost
        # PROSSIMA RIGA AD RISCRIVERE!!!
        path_cost = (jnp.sum(jnp.array([jnp.sum(i) for i in 
                                        jnp.array(shortest_distance(NetPar.G,directed=True))])  
                            )/norm_factor)**2          # average shortest path length cost
        if verb > 0:
            jdb.print('Target distance:{d}',d=target_dist)
            jdb.print('Volume cost:{d}',d=volume_cost)
            jdb.print('Length cost:{d}',d=length_cost)
            jdb.print('Path cost:{d}',d=path_cost)
        fitness += jnp.exp(target_dist + volume_cost + length_cost + path_cost) # take the exponential of the sum of the costs (weighted if needed)
        return fitness

In [ ]:
key, _ = split(key)



Now I define the genetic algorithm for the evolution:

In [ ]:
############# GENETIC ALGORITHM #################
def crossover(par1:Network,par2:Network,par:dict):      # PER ORA IMPLEMENTO SOLO PER CChromo
    # Implements the crossover ricombination given 2 parent Networks
    # and returns 2 offspring Networks.
    if len(par1.Cchromo) != len(par2.Cchromo):
        jdb.print('Error: length mismatch between the chromosomes of the two parents!')
        raise ValueError
    offspring
    subkey, par = gen_key(par)
    cut_idx = jrd.choice(subkey,jnp.arange(len(par1.Cchromo)))  # randomly pick where to cut the chromosomes
    

def evolution(par:dict, verb:int=1,early_stop:bool=True):
    if par(['N_sol']) % 2 != 0:     # N_sol must be even
        jdb.print('Error:The number of solutions N_sol must be an even positive number.')
        raise ValueError
    # Generate the initial batch of solutions
    solutions = []
    for _ in range(par['N_sol']):
        # Generate a solution
        n = Network()
        n.generate(par)     # already computes also the fitness value
        solutions.append(n)
    solutions = jnp.array(solutions)
    # Initiate some container for statistic
    Fmean_values = []
    
    ##  EVOLUTION
    for iter in range(par['n_iter']):
        # Compute the mean fitness for statistics
        mean_fit = jnp.sum(jnp.array([sol.fitness for sol in solutions])) / par['N_sol']    # Here I don't have to divide also by 4, because I've already done it in the compute_fitness method
        # Save the old solution set for possible early stopping
        solutions_old = []
        for sol in solutions:
            solutions_old.append(copy.deepcopy(sol))
        # SELECTION
        solutions = jnp.sort(solutions,key=lambda sol: sol.fitness)    # sort in ascending order based on fitness
        subkey, par = gen_key(par)
        parents_idx = jrd.choice(subkey,jnp.arange(par['N_sol']),                       # must be with replacement, since the same individual 
                                 shape=(par['N_sol']/2,2),replace=True,                 # can be chosen to be parent multiple times
                                 p=jnp.array([1/sol.fitness for sol in sols])           # p: probability of being chosen as a parent, inversly proportional to the fitness
                                 /jnp.sum(jnp.array([sol.fitness for sol in sols]))) 
        
        
        